# 05 - Fusion + Graph Neural Network

**Architecture blocks 3-4.** Vision (+) climate (+) metadata -> fusion -> GNN
-> spatially aware node embeddings.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent / "src"))

import cropforecast
from cropforecast.config import load_config, ensure_dirs, set_seed, Device
cfg = load_config(Path.cwd().parent / "configs" / "default.yaml")
ensure_dirs(cfg); set_seed(cfg.project.seed)
device = Device.auto(cfg.training.amp)
print("device:", device)

## Fusion

Two modes. `concat` is the diagram's own concatenation/projection. `gated` adds
FiLM modulation so climate can *scale* the vision stream - the same lesion means
something different in a wet spell than a dry one.

In [ ]:
import torch
from cropforecast.models.fusion import FusionModule
f = FusionModule(vision_dim=384, climate_dim=51, meta_dim=30, hidden_dim=256, mode="gated")
v, c, m = torch.randn(8,384), torch.randn(8,51), torch.randn(8,30)
print("fused:", tuple(f(v,c,m).shape), "| params:", sum(p.numel() for p in f.parameters())/1e3, "K")

## Graph encoder

All three convolutions from the diagram, plus a no-graph ablation.

In [ ]:
from cropforecast.models.gnn import build_encoder
x = torch.randn(200, 256); ei = torch.randint(0, 200, (2, 900)); ea = torch.rand(900, 4)
for conv in ["sage", "gcn", "gat", "none"]:
    enc = build_encoder(conv, 256, hidden_dim=256, num_layers=3, heads=4)
    print(f"{conv:5s} -> {tuple(enc(x, ei, ea).shape)}   params {sum(p.numel() for p in enc.parameters())/1e3:7.1f} K")

## The full model

In [ ]:
from cropforecast.models.full_model import CropDiseaseForecastNet
net = CropDiseaseForecastNet(vision_dim=384, climate_dim=51, meta_dim=30,
                             num_classes=38, horizons=(1,3,5,7), gnn_conv="sage")
out = net(torch.randn(200,384), torch.randn(200,51), torch.randn(200,30), ei, ea)
for k, val in out.items(): print(f"  {k:14s} {tuple(val.shape)}")
print("trainable:", round(net.num_trainable()/1e6, 3), "M  (backbone is frozen and cached)")

### Why the forecaster is autoregressive

Horizon *h* is predicted from the state carried forward from *h-1* through a GRU
cell. Independent heads have no way to know day 7 follows day 5 and produce
forecasts that jump around non-physically.

In [ ]:
r = out["risk"][:5].detach()
import pandas as pd
pd.DataFrame(r.numpy(), columns=["+1d","+3d","+5d","+7d"]).round(3)

### Ablation results

Run `python scripts/04_train_and_ablate.py` first.

In [ ]:
abl = Path(cfg.paths.reports) / "stage4_ablations.csv"
pd.read_csv(abl) if abl.exists() else print("Run scripts/04_train_and_ablate.py")